# Mockdata

## Objetivo

Proveer un DSL interpretable para la creación de datasets mock(falsos) en formato JSON.

## Alcance 

* Permite generar colecciones de N entidades (usuarios, productos, etc) mediante la instrucción GENERAR.
* Provee tipos de datos predefinidos orientados al desarrollo web: NOMBRE, APELLIDO, EMAIL, CIUDAD, EDAD, NUMERO, BOOLEAN, TEXTO, TELEFONO, FECHA, entre otros.
* Incorpora modificadores para personalizar la generación: rango numérico (ENTRE a Y b), probabilidad porcentual para booleanos (80%) entre otros.
* Permite exportar los datos generados en formatos estándar para frontend (JSON) mediante la instrucción GUARDAR EN ... COMO.
* Incluye validación semántica para detectar inconsistencias como rangos invertidos (ENTRE 65 Y 18) o formatos de archivo no soportados.

* Está inspirado en herramientas como Mockaroo y Faker, pero a diferencia de Mockaroo que es una interfaz web, este traductor es un DSL (Domain Specific Language) textual que permite generar datos sintéticos de forma reproducible y versionable, con restricciones como UNICO y REFERENCIA para datos relacionales.

## Especificaciones léxicas

| Token | Lexema (ejemplo) | Patrón (Regex) |
| :--- | :--- | :--- |
| GENERAR | GENERAR | `GENERAR` |
| GUARDAR | GUARDAR | `GUARDAR` |
| RAIZ | RAIZ | `RAIZ` |
| REFERENCIA | REFERENCIA | `REFERENCIA` |
| UNICO | UNICO | `UNICO` |
| ENTRE | ENTRE | `ENTRE` |
| EN | EN | `EN` |
| COMO | COMO | `COMO` |
| Y | Y | `Y` |
| TIPO_DATO | NOMBRE, EMAIL, DNI | `NOMBRE\|APELLIDO\|EMAIL\|EDAD\|NUMERO\|BOOLEAN\|CIUDAD\|TELEFONO\|FECHA\|DNI` |
| TIPO_ARCHIVO | json | `json\|csv\|sql` |
| NUMERO | 10, 1234 | `\d+` |
| STRING | "usuarios.json" | `"[^"]*"` |
| ID | usuarios, nombre, dni | `[a-z_][a-z0-9_]*` |
| DOS_PUNTOS | : | `:` |
| PORCENTAJE | % | `%` |



## Especificaciones sintácticas

```bnf
<mock> ::= <generar> <resto>
<resto> ::= <generar> <resto> | <guardar> <resto> | λ

<generar> ::= GENERAR NUMERO IDENTIFICADOR <raiz_op> <bloque_campos>

<raiz_op> ::= RAIZ NUMERO | λ

<bloque_campos> ::= <campo> | <campo> <bloque_campos>

<campo> ::= IDENTIFICADOR DOS_PUNTOS <tipo> <modificador_op>
          | IDENTIFICADOR DOS_PUNTOS REFERENCIA IDENTIFICADOR <modificador_op>

<tipo> ::= NOMBRE | APELLIDO | EMAIL | EDAD | NUMERO | BOOLEAN | CIUDAD | TELEFONO | FECHA | DNI

<modificador_op> ::= <modificador> | λ

<modificador> ::= ENTRE NUMERO Y NUMERO
                | NUMERO PORCENTAJE
                | NUMERO
                | UNICO

<guardar> ::= GUARDAR EN STRING COMO TIPO_ARCHIVO
```

## Especificaciones semánticas

| Regla Sintáctica (GIC) | Acción Semántica |
| :--- | :--- |
| `<mock> ::= <generar> <resto>` | Inicializa entorno, crea tabla de símbolos vacía, ejecuta generar y luego resto. Garantiza que empieza con GENERAR. |
| `<resto> ::= <generar> <resto>` | Ejecuta otro GENERAR y sigue. |
| `<resto> ::= <guardar> <resto>` | Despacha a GUARDAR. Verifica que tabla no esté vacía, si está vacía -> error semántico. |
| `<resto> ::= λ` | Fin del programa. |
| `<generar> ::= GENERAR NUMERO IDENTIFICADOR <raiz_op> <bloque_campos>` | Valida NUMERO>0, ejecuta raiz_op, crea caja en tabla, repite NUMERO veces, guarda lista. |
| `<raiz_op> ::= RAIZ NUMERO` | Fija semilla: random.seed(NUMERO). |
| `<raiz_op> ::= λ` | lambda. |
| `<bloque_campos> ::= <campo>` | Inicializa lista de campos. |
| `<bloque_campos> ::= <campo> <bloque_campos>` | Acumula campo, verifica no duplicado. |
| `<campo> ::= IDENTIFICADOR DOS_PUNTOS <tipo> <modificador_op>` | Arma fichita {nombre, tipo, mod, ref=None}. |
| `<campo> ::= IDENTIFICADOR DOS_PUNTOS REFERENCIA IDENTIFICADOR <modificador_op>` | Verifica que IDENTIFICADOR referenciado exista, si no -> error semántico. Arma fichita con ref. |
| `<tipo> ::= NOMBRE \| ... \| DNI` | Retorna tipo y genera valor. |
| `<modificador_op> ::= <modificador>` | Retorna modificador. |
| `<modificador_op> ::= λ` | Retorna None. |
| `<modificador> ::= ENTRE NUMERO Y NUMERO` | Valida NUM1 < NUM2. Retorna rango. |
| `<modificador> ::= NUMERO PORCENTAJE` | Valida 0-100. Retorna porcentaje. |
| `<modificador> ::= NUMERO` | Valida >0. Retorna cantidad. |
| `<modificador> ::= UNICO` | Marca único, verifica duplicados. |
| `<guardar> ::= GUARDAR EN STRING COMO TIPO_ARCHIVO` | Verifica entidad en tabla, valida tipo archivo, escribe archivo. |

## Implementaciones

In [1]:
import ply.lex as lex

tokens = (
    'GENERAR','GUARDAR','RAIZ','REFERENCIA','UNICO','ENTRE','EN','COMO','Y',
    'TIPO_DATO','TIPO_ARCHIVO','NUMERO','STRING','IDENTIFICADOR','DOS_PUNTOS','PORCENTAJE',
)

t_GENERAR = r'GENERAR'
t_GUARDAR = r'GUARDAR'
t_RAIZ = r'RAIZ'
t_REFERENCIA = r'REFERENCIA'
t_UNICO = r'UNICO'
t_ENTRE = r'ENTRE'
t_EN = r'EN'
t_COMO = r'COMO'
t_Y = r'Y'
t_TIPO_DATO = r'NOMBRE|APELLIDO|EMAIL|EDAD|NUMERO|BOOLEAN|CIUDAD|TELEFONO|FECHA|DNI'
t_TIPO_ARCHIVO = r'json'
t_DOS_PUNTOS = r':'
t_PORCENTAJE = r'%'

def t_NUMERO(t):
    r'\d+'
    t.value = int(t.value)
    return t

def t_STRING(t):
    r'"[^"]*"'
    t.value = t.value.strip('"')
    return t

def t_IDENTIFICADOR(t):
    r'[a-z_][a-z0-9_]*'
    return t

t_ignore = ' \t\n'

def t_error(t):
    raise Exception(f"Error lexico en posicion {t.lexpos}: '{t.value[0]}'")

lexer = lex.lex()


TypeError: <module '__main__'> is a built-in module

In [2]:
import json
import random

tabla_simbolos = {}

def generar_valor(tipo, modificador=None):
    if modificador and modificador.get('tipo') == 'rango':
        return random.randint(modificador['min'], modificador['max'])
    if tipo == 'BOOLEAN' and modificador and modificador.get('tipo') == 'porcentaje':
        return random.random() * 100 < modificador['valor']
    if tipo == 'NOMBRE':
        return random.choice(["Manuel", "Axel", "Maximiliano", "Loana", "Lucas", "Pedro", "Ramon", "Sofia", "Ana", "Florencia"])
    if tipo == 'APELLIDO':
        return random.choice(["Acuña", "Castro", "Lopez", "Fernandez", "Gonzalez", "Cabañas", "Costa", "Perez", "Sosa", "Arbizu"])
    if tipo == 'EMAIL':
        return f"user{random.randint(1,9999)}@gmail.com"
    if tipo == 'EDAD':
        return random.randint(18, 65)
    if tipo == 'NUMERO':
        return random.randint(1, 100)
    if tipo == 'BOOLEAN':
        return random.choice([True, False])
    if tipo == 'CIUDAD':
        return random.choice(["Buenos Aires", "Cordoba", "Rosario", "Mendoza", "Entre Rios", "Misiones", "Santa Cruz", "Neuquen", "Tucuman", "Chaco"])
    if tipo == 'TELEFONO':
        return f"11-{random.randint(1000,9999)}-{random.randint(1000,9999)}"
    if tipo == 'FECHA':
        return f"2026-{random.randint(1,12):02d}-{random.randint(1,28):02d}"
    if tipo == 'DNI':
        return random.randint(20000000, 50000000)
    return f"<{tipo}>"

def interpretar(codigo):
    global tabla_simbolos
    tabla_simbolos = {} 
    tokens = lexer(codigo)
    
    if not tokens or tokens[0][0] != 'GENERAR':
        raise Exception("ERROR SINTACTICO: <mock> debe empezar con GENERAR, no puede empezar con GUARDAR")
    
    i = 0
    while i < len(tokens):
        if tokens[i][0] == 'GENERAR':
            cantidad = int(tokens[i+1][1])
            if cantidad <= 0:
                raise Exception("Error SEMANTICO: NUMERO debe ser > 0")
            entidad = tokens[i+2][1]
            i += 3
            
            if i < len(tokens) and tokens[i][0] == 'RAIZ':
                semilla = int(tokens[i+1][1])
                random.seed(semilla)  
                i += 2
            
            campos = [] 
            while i < len(tokens) and tokens[i][0] == 'IDENTIFICADOR':
                nombre_campo = tokens[i][1]
              
                i += 2 
                
                if i < len(tokens) and tokens[i][0] == 'REFERENCIA':
                    ref_entidad = tokens[i+1][1]
                    if ref_entidad not in tabla_simbolos:
                        raise Exception(f"[SEMANTICO] Error: referencia inexistente {ref_entidad}")
                    mod = None
                    unico = False
                    if i+2 < len(tokens) and tokens[i+2][0] == 'UNICO':
                        unico = True
                        i += 3
                    else:
                        i += 2
                    campos.append((nombre_campo, 'REFERENCIA', None, ref_entidad, unico))
                    continue
                
                tipo = tokens[i][1]
                i += 1
                mod = None
                unico = False
                
                if i < len(tokens) and tokens[i][0] == 'ENTRE':
                    min_v = int(tokens[i+1][1])
                    max_v = int(tokens[i+3][1])
                    if not min_v < max_v:
                        raise Exception("Error SEMANTICO: ENTRE min debe ser < max")
                    mod = {'tipo': 'rango', 'min': min_v, 'max': max_v}
                    i += 4 
                elif i < len(tokens) and tokens[i][0] == 'NUMERO' and i+1 < len(tokens) and tokens[i+1][0] == 'PORCENTAJE':
                    val = int(tokens[i][1])
                    if not 0 <= val <= 100:
                        raise Exception("Error SEMANTICO: porcentaje 0-100")
                    mod = {'tipo': 'porcentaje', 'valor': val}
                    i += 2
                elif i < len(tokens) and tokens[i][0] == 'NUMERO':
                    i += 1
                elif i < len(tokens) and tokens[i][0] == 'UNICO':
                    unico = True
                    i += 1
                campos.append((nombre_campo, tipo, mod, None, unico))

            if len(campos) == 0:
                raise Exception("Error SEMANTICO: GENERAR debe tener al menos un campo, por ejemplo nombre: NOMBRE")

            lista = []
            usados = set()  
            for _ in range(cantidad):
                obj = {}
                for nombre_campo, tipo, mod, ref_entidad, unico in campos:
                    if tipo == 'REFERENCIA':
                        ref_lista = tabla_simbolos[ref_entidad]
                        elegido = random.choice(ref_lista)
                        valor = list(elegido.values())[0]
                    else:
                        valor = generar_valor(tipo, mod)
                        if unico:
                            intentos = 0
                            while valor in usados and intentos < 100:
                                valor = generar_valor(tipo, mod)
                                intentos += 1
                            if valor in usados:
                                raise Exception(f"Error SEMANTICO: no se pudo generar UNICO para {nombre_campo}")
                            usados.add(valor)
                    obj[nombre_campo] = valor
                lista.append(obj)
            tabla_simbolos[entidad] = lista
            continue
            
        if tokens[i][0] == 'GUARDAR':
            if not tabla_simbolos:
                raise Exception("Error SEMANTICO: No podes GUARDAR sin GENERAR antes. Tabla vacia")
            
            nombre_archivo = tokens[i+2][1].strip('"')
            tipo_archivo = tokens[i+4][1]
            if tipo_archivo != 'json':
                raise Exception("Error SEMANTICO: solo se permite json")
            if not nombre_archivo.endswith('.json'):
                raise Exception("Error SEMANTICO: extension debe ser .json")
           
            ultima = list(tabla_simbolos.values())[-1]
            with open(nombre_archivo, 'w', encoding='utf-8') as f:
                json.dump(ultima, f, indent=2, ensure_ascii=False)
            print(f"Archivo {nombre_archivo} guardado con {len(ultima)} objetos")
            i += 5
            continue
        i += 1
    return tabla_simbolos

### **Pruebas**:

#### Generar 3 usuarios con solo nombre


**ANALISIS LEXICO**

GENERAR 3 usuarios

  nombre: NOMBRE
  
GUARDAR EN "usuarios.json" COMO json

**ANALISIS SINTACTICO**
```bnf
<mock> 
<mock>
=> <generar> <resto>
=> GENERAR NUMERO IDENTIFICADOR <raiz_op> <bloque_campos> <resto>
=> GENERAR 3 usuarios <raiz_op> <bloque_campos> <resto>
=> GENERAR 3 usuarios λ <bloque_campos> <resto>
=> GENERAR 3 usuarios λ <campo> <resto>
=> GENERAR 3 usuarios λ IDENTIFICADOR DOS_PUNTOS <tipo> <modificador_op> <resto>
=> GENERAR 3 usuarios λ nombre DOS_PUNTOS NOMBRE <modificador_op> <resto>
=> GENERAR 3 usuarios λ nombre DOS_PUNTOS NOMBRE λ <resto>
=> GENERAR 3 usuarios λ nombre DOS_PUNTOS NOMBRE λ <guardar> <resto>
=> GENERAR 3 usuarios λ nombre DOS_PUNTOS NOMBRE λ GUARDAR EN STRING COMO TIPO_ARCHIVO <resto>
=> GENERAR 3 usuarios λ nombre DOS_PUNTOS NOMBRE λ GUARDAR EN "usuarios.json" COMO json <resto>
=> GENERAR 3 usuarios λ nombre DOS_PUNTOS NOMBRE λ GUARDAR EN "usuarios.json" COMO json λ
=> GENERAR 3 usuarios nombre : NOMBRE GUARDAR EN "usuarios.json" COMO json
```

In [3]:
# Prueba Ejercicio 1
codigo_ej1 = """
GENERAR 3 usuarios
  nombre: NOMBRE
GUARDAR EN "usuarios.json" COMO json
"""
interpretar(codigo_ej1)

Archivo usuarios.json guardado con 3 objetos


{'usuarios': [{'nombre': 'Maximiliano'},
  {'nombre': 'Sofia'},
  {'nombre': 'Ana'}]}

#### Generar 5 usuarios y sus respectivos DNI's con SEMILLA.


```bnf
<mock>
=> <generar> <resto>
=> GENERAR NUMERO IDENTIFICADOR <raiz_op> <bloque_campos> <resto>
=> GENERAR 5 usuarios <raiz_op> <bloque_campos> <resto>
=> GENERAR 5 usuarios RAIZ 1234 <bloque_campos> <resto>
=> GENERAR 5 usuarios RAIZ 1234 <campo> <bloque_campos> <resto>
=> GENERAR 5 usuarios RAIZ 1234 IDENTIFICADOR DOS_PUNTOS <tipo> <modificador_op> <bloque_campos> <resto>
=> GENERAR 5 usuarios RAIZ 1234 nombre DOS_PUNTOS NOMBRE λ <bloque_campos> <resto>
=> GENERAR 5 usuarios RAIZ 1234 nombre DOS_PUNTOS NOMBRE λ <campo> <resto>
=> GENERAR 5 usuarios RAIZ 1234 nombre DOS_PUNTOS NOMBRE λ IDENTIFICADOR DOS_PUNTOS <tipo> <modificador_op> <resto>
=>  GENERAR 5 usuarios RAIZ 1234 nombre DOS_PUNTOS NOMBRE λ dni DOS_PUNTOS <tipo> <modificador_op> <resto> 
=>  GENERAR 5 usuarios RAIZ 1234 nombre DOS_PUNTOS NOMBRE λ dni DOS_PUNTOS DNI <modificador_op> <resto> 
=>  GENERAR 5 usuarios RAIZ 1234 nombre DOS_PUNTOS NOMBRE λ dni DOS_PUNTOS DNI <modificador_op> <resto>  
=>  GENERAR 5 usuarios RAIZ 1234 nombre DOS_PUNTOS NOMBRE λ dni DOS_PUNTOS DNI UNICO <resto>
=>  GENERAR 5 usuarios RAIZ 1234 nombre DOS_PUNTOS NOMBRE λ dni DOS_PUNTOS DNI UNICO <guardar> <resto>
=>  GENERAR 5 usuarios RAIZ 1234 nombre DOS_PUNTOS NOMBRE λ dni DOS_PUNTOS DNI UNICO GUARDAR EN STRING COMO TIPO_ARCHIVO <resto>
=>  GENERAR 5 usuarios RAIZ 1234 nombre DOS_PUNTOS NOMBRE λ dni DOS_PUNTOS DNI UNICO GUARDAR EN "raiz.json" COMO json <resto>
=>  GENERAR 5 usuarios RAIZ 1234 nombre DOS_PUNTOS NOMBRE λ dni DOS_PUNTOS DNI UNICO GUARDAR EN "raiz.json" COMO json λ

```

In [4]:
codigo_raiz = """
GENERAR 5 usuarios RAIZ 1234
  nombre: NOMBRE
  dni: DNI UNICO
GUARDAR EN "raiz.json" COMO json
"""
interpretar(codigo_raiz)

Archivo raiz.json guardado con 5 objetos


{'usuarios': [{'nombre': 'Sofia', 'dni': 23920852},
  {'nombre': 'Manuel', 'dni': 23041494},
  {'nombre': 'Florencia', 'dni': 21171927},
  {'nombre': 'Axel', 'dni': 23302495},
  {'nombre': 'Pedro', 'dni': 27946017}]}

In [6]:
prueba03 = """
GENERAR 5 usuarios RAIZ 1234
nombre: NOMBRE
dni: DNI UNICO
GUARDAR EN "prueba03.json" COMO json
"""
interpretar(prueba03)

Archivo prueba03.json guardado con 5 objetos


{'usuarios': [{'nombre': 'Sofia', 'dni': 23920852},
  {'nombre': 'Manuel', 'dni': 23041494},
  {'nombre': 'Florencia', 'dni': 21171927},
  {'nombre': 'Axel', 'dni': 23302495},
  {'nombre': 'Pedro', 'dni': 27946017}]}

In [5]:
prueba04 = """
GENERAR 5 usuarios
nombre: NOMBRE
dni: DNI UNICO
edad:  NUMERO ENTRE 14 Y 32 
GUARDAR EN "prueba04.json" COMO json
"""
interpretar(prueba04)

Archivo prueba04.json guardado con 5 objetos


{'usuarios': [{'nombre': 'Manuel', 'dni': 21033946, 'edad': 14},
  {'nombre': 'Pedro', 'dni': 41753876, 'edad': 29},
  {'nombre': 'Florencia', 'dni': 35554088, 'edad': 18},
  {'nombre': 'Axel', 'dni': 26143502, 'edad': 17},
  {'nombre': 'Manuel', 'dni': 36880687, 'edad': 29}]}

In [6]:
ej_ext = """
GENERAR 5 empleados
nombre: NOMBRE 
apellido: APELLIDO
dni: DNI UNICO
tel: TELEFONO UNICO
email: EMAIL UNICO
ciudad: CIUDAD
edad: NUMERO ENTRE 18 Y 65
activo: BOOLEAN 80%
GUARDAR EN "ext.json" COMO json
"""
interpretar(ej_ext)

Archivo ext.json guardado con 5 objetos


{'empleados': [{'nombre': 'Loana',
   'apellido': 'Castro',
   'dni': 42389380,
   'tel': '11-9865-8633',
   'email': 'user1124@gmail.com',
   'ciudad': 'Chaco',
   'edad': 61,
   'activo': True},
  {'nombre': 'Florencia',
   'apellido': 'Acuña',
   'dni': 29045788,
   'tel': '11-2033-5954',
   'email': 'user7882@gmail.com',
   'ciudad': 'Misiones',
   'edad': 35,
   'activo': True},
  {'nombre': 'Loana',
   'apellido': 'Perez',
   'dni': 49412295,
   'tel': '11-9686-1316',
   'email': 'user8325@gmail.com',
   'ciudad': 'Cordoba',
   'edad': 18,
   'activo': True},
  {'nombre': 'Sofia',
   'apellido': 'Costa',
   'dni': 25607363,
   'tel': '11-1887-2249',
   'email': 'user9627@gmail.com',
   'ciudad': 'Neuquen',
   'edad': 28,
   'activo': True},
  {'nombre': 'Loana',
   'apellido': 'Lopez',
   'dni': 20910286,
   'tel': '11-6672-2816',
   'email': 'user8679@gmail.com',
   'ciudad': 'Rosario',
   'edad': 60,
   'activo': True}]}

In [7]:
vacio =  """
GENERAR 3 usuarios
GUARDAR EN "usuarios.json" COMO json
"""
interpretar(vacio)

Exception: Error SEMANTICO: GENERAR debe tener al menos un campo, por ejemplo nombre: NOMBRE